# La forma del día · datos nuevos (Semana 12)

Tres pasadas. Cada una escribe un CSV; al final se empaquetan en `datos_nuevos.zip`.

| | Qué trae | Para qué | Tiempo |
|---|---|---|---|
| **D** | hora-de-la-semana de **enero–febrero 2024** | la misma ventana calendario que la línea base: corrige el sesgo estacional | ~1 min |
| **E** | estación × mes × hora × laborable/finde, los 60 meses | ver cómo cambia la forma del día mes a mes; el brush temporal reconstruye perfiles | ~10 min |
| **F** | Census ACS por área de influencia de cada estación | teletrabajo, ingreso, densidad: la capa que explica el *por qué* | ~3 min |

Las pasadas son resumibles: si una celda se corta, vuelve a correrla y continúa donde quedó.

In [ ]:
# ── 1 · configuración ──────────────────────────────────────────────────────
SOCRATA_APP_TOKEN = ""     # ← pega acá tu token. No lo subas al repo.

DATASET_ID = "wujg-7c2s"
RESOURCE_URL = f"https://data.ny.gov/resource/{DATASET_ID}.json"
TIMEOUT, PAGE, WINDOW_DAYS = 150, 50000, 7

import json, os, time, csv, urllib.parse, urllib.request
from collections import defaultdict
from datetime import date, timedelta

def headers():
    h = {"User-Agent": "data-visualization-2026-2/1.0"}
    if SOCRATA_APP_TOKEN: h["X-App-Token"] = SOCRATA_APP_TOKEN
    return h

def fetch(params, retries=3, quiet=False):
    url = RESOURCE_URL + "?" + urllib.parse.urlencode(params); delay = 4.0
    for attempt in range(1, retries + 1):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=headers()), timeout=TIMEOUT) as r:
                return json.loads(r.read().decode("utf-8"))
        except Exception as exc:
            if attempt == retries:
                if not quiet: print(f"      fallo: {exc}")
                return None
            time.sleep(delay); delay *= 2

def query_range(select, group, ini, fin, order=None):
    out, offset = [], 0
    while True:
        p = {"$select": select, "$group": group, "$limit": PAGE, "$offset": offset,
             "$where": f"transit_timestamp >= '{ini.isoformat()}T00:00:00' AND transit_timestamp < '{fin.isoformat()}T00:00:00'"}
        if order: p["$order"] = order
        page = fetch(p, quiet=(offset > 0))
        if page is None: return None
        out.extend(page)
        if len(page) < PAGE: return out
        offset += PAGE

def query_window(select, group, ini, fin, order=None):
    r = query_range(select, group, ini, fin, order)
    if r is not None: return r
    print("      ventana lenta, día por día ...", end=" ", flush=True)
    out, cur = [], ini
    while cur < fin:
        d = query_range(select, group, cur, cur + timedelta(days=1), order)
        if d is None: return None
        out.extend(d); cur += timedelta(days=1)
    print("ok"); return out

def num(x):
    try: return float(x)
    except (TypeError, ValueError): return 0.0

SEL_B = ("station_complex_id, date_extract_dow(transit_timestamp) AS dow, "
         "date_extract_hh(transit_timestamp) AS hora, sum(ridership) AS ridership")
GRP_B = "station_complex_id, date_extract_dow(transit_timestamp), date_extract_hh(transit_timestamp)"

print("Token:", "sí" if SOCRATA_APP_TOKEN else "NO — ponlo antes de seguir")

## Pasada D · hora-de-la-semana, enero–febrero de 2024

Misma consulta que la línea base de la Semana 6, sobre la misma ventana calendario cuatro años después.

In [ ]:
D, D_done = {}, set()
ini0, fin0 = date(2024, 1, 1), date(2024, 3, 1)
ventanas = []
cur = ini0
while cur < fin0:
    nxt = min(cur + timedelta(days=WINDOW_DAYS), fin0); ventanas.append((cur, nxt)); cur = nxt

t0 = time.time()
for k, (ini, fin) in enumerate(ventanas, 1):
    if ini in D_done: continue
    data = query_window(SEL_B, GRP_B, ini, fin, "station_complex_id")
    if data is None: print(f"  {ini} FALLO"); continue
    for r in data:
        cid = r.get("station_complex_id")
        if not cid: continue
        key = (cid, int(num(r.get("dow"))), int(num(r.get("hora"))))
        D[key] = D.get(key, 0.0) + num(r.get("ridership"))
    D_done.add(ini)
    print(f"  {k}/{len(ventanas)}  {ini}  {len(D):,} filas  ({time.time()-t0:.0f}s)")

with open("hourofweek_2024_enefeb.csv", "w", newline="") as fh:
    w = csv.writer(fh); w.writerow(["station_complex_id", "dow", "hora", "ridership"])
    for (cid, dow, hora), v in sorted(D.items()): w.writerow([cid, dow, hora, int(v)])
print(f"\nhourofweek_2024_enefeb.csv  {os.path.getsize('hourofweek_2024_enefeb.csv')/1e6:.2f} MB  ·  {len(D_done)}/{len(ventanas)} ventanas")

## Pasada E · estación × mes × hora × tipo de día, 2020–2024

Ventanas de 7 días **alineadas al mes** (1–7, 8–14, 15–21, 22–fin) para que ninguna cruce un cambio de mes.
Se agrega `dow` a laborable (1–5) / fin de semana (0, 6). Unos 265 ventanas, ~10 minutos.

In [ ]:
E, E_done = {}, set()
ventanas = []
for y in range(2020, 2025):
    for m in range(1, 13):
        ini_m = date(y, m, 1); fin_m = date(y + 1, 1, 1) if m == 12 else date(y, m + 1, 1)
        cur = ini_m
        while cur < fin_m:
            nxt = min(cur + timedelta(days=WINDOW_DAYS), fin_m); ventanas.append((f"{y}-{m:02d}", cur, nxt)); cur = nxt
print(len(ventanas), "ventanas")

t0 = time.time()
for k, (mes, ini, fin) in enumerate(ventanas, 1):
    if ini in E_done: continue
    data = query_window(SEL_B, GRP_B, ini, fin, "station_complex_id")
    if data is None: print(f"  {ini} FALLO"); continue
    for r in data:
        cid = r.get("station_complex_id")
        if not cid: continue
        dow = int(num(r.get("dow"))); lab = 0 if dow in (0, 6) else 1
        key = (cid, mes, lab, int(num(r.get("hora"))))
        E[key] = E.get(key, 0.0) + num(r.get("ridership"))
    E_done.add(ini)
    if k % 20 == 0 or k == len(ventanas):
        print(f"  {k}/{len(ventanas)}  {ini}  {len(E):,} filas  ({time.time()-t0:.0f}s)")

import gzip
with gzip.open("station_month_hour.csv.gz", "wt", newline="") as fh:
    w = csv.writer(fh); w.writerow(["station_complex_id", "mes", "laborable", "hora", "ridership"])
    for (cid, mes, lab, hora), v in sorted(E.items()): w.writerow([cid, mes, lab, hora, int(v)])
print(f"\nstation_month_hour.csv.gz  {os.path.getsize('station_month_hour.csv.gz')/1e6:.2f} MB  ·  {len(E_done)}/{len(ventanas)} ventanas")

## Pasada F · Census ACS 2018–2022 por área de influencia de cada estación

Para cada estación, un círculo de **800 m** (≈10 minutos a pie). Se intersecta con los tracts censales de 2020
y se pondera por área. Variables:

| | código ACS |
|---|---|
| trabajadores de 16+ | `B08301_001E` |
| que van al trabajo en transporte público | `B08301_010E` |
| que **trabajan desde casa** | `B08301_021E` |
| ingreso mediano del hogar | `B19013_001E` |
| población total | `B01003_001E` |

`stations_coords.csv` se baja solo del repo si no está en la carpeta.

In [ ]:
import io, zipfile, urllib.request
import pandas as pd, geopandas as gpd
from shapely.geometry import Point

# --- tracts 2020 de los cinco condados de NYC (TIGER/Line 2022)
TIGER = "https://www2.census.gov/geo/tiger/TIGER2022/TRACT/tl_2022_36_tract.zip"
if not os.path.exists("tl_2022_36_tract.shp"):
    print("bajando TIGER ...", end=" ", flush=True)
    zf = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(TIGER, timeout=300).read())); zf.extractall("."); print("ok")
NYC = ["005", "047", "061", "081", "085"]   # Bronx, Kings, New York, Queens, Richmond
tr = gpd.read_file("tl_2022_36_tract.shp"); tr = tr[tr["COUNTYFP"].isin(NYC)].copy()
tr["GEOID"] = tr["GEOID"].astype(str)
print(len(tr), "tracts")

# --- ACS 5-year 2022
VARS = "B08301_001E,B08301_010E,B08301_021E,B19013_001E,B01003_001E"
filas = []
for c in NYC:
    url = f"https://api.census.gov/data/2022/acs/acs5?get={VARS}&for=tract:*&in=state:36&in=county:{c}"
    data = json.loads(urllib.request.urlopen(url, timeout=120).read().decode())
    cols = data[0]
    for row in data[1:]: filas.append(dict(zip(cols, row)))
acs = pd.DataFrame(filas)
acs["GEOID"] = acs["state"] + acs["county"] + acs["tract"]
for v in VARS.split(","): acs[v] = pd.to_numeric(acs[v], errors="coerce")
acs.loc[acs["B19013_001E"] < 0, "B19013_001E"] = pd.NA        # -666666666 = sin dato
tr = tr.merge(acs[["GEOID"] + VARS.split(",")], on="GEOID", how="left")
print(len(acs), "tracts con ACS")

# --- estaciones (si no está el CSV, se baja del repo)
if not os.path.exists("stations_coords.csv"):
    RAW = "https://raw.githubusercontent.com/stewartmb/data-visualization-2026-2/main/deliveries/week06/data/processed/stations_coords.csv"
    open("stations_coords.csv", "wb").write(urllib.request.urlopen(RAW, timeout=60).read()); print("stations_coords.csv bajado del repo")
st = pd.read_csv("stations_coords.csv")
st = st.dropna(subset=["latitude", "longitude"])
g = gpd.GeoDataFrame(st, geometry=[Point(xy) for xy in zip(st["longitude"], st["latitude"])], crs="EPSG:4326").to_crs(2263)
g["geometry"] = g.geometry.buffer(800 / 0.3048)
tr2 = tr.to_crs(2263); tr2["area_tr"] = tr2.geometry.area

inter = gpd.overlay(g[["station_complex_id", "geometry"]], tr2, how="intersection")
inter["w"] = inter.geometry.area / inter["area_tr"]          # fracción del tract dentro del círculo
for v in ["B08301_001E", "B08301_010E", "B08301_021E", "B01003_001E"]:
    inter[v + "_w"] = inter[v].fillna(0) * inter["w"]
inter["ing_w"] = inter["B19013_001E"].fillna(0) * inter["B01003_001E_w"]      # ingreso ponderado por población
inter["ing_pop"] = inter["B01003_001E_w"].where(inter["B19013_001E"].notna(), 0)
inter["area_km2"] = inter.geometry.area * 0.3048**2 / 1e6

agg = inter.groupby("station_complex_id").agg(
    workers=("B08301_001E_w", "sum"), transit=("B08301_010E_w", "sum"), wfh=("B08301_021E_w", "sum"),
    pob=("B01003_001E_w", "sum"), ing_w=("ing_w", "sum"), ing_pop=("ing_pop", "sum"), area_km2=("area_km2", "sum")).reset_index()
agg["wfh_share"] = agg["wfh"] / agg["workers"]
agg["transit_share"] = agg["transit"] / agg["workers"]
agg["ingreso_mediano"] = agg["ing_w"] / agg["ing_pop"]
agg["densidad_pob"] = agg["pob"] / agg["area_km2"]
out = agg[["station_complex_id", "wfh_share", "transit_share", "ingreso_mediano", "densidad_pob", "pob", "workers"]].round(4)
out.to_csv("acs_por_estacion.csv", index=False)
print(out.describe().round(3))
print("\nacs_por_estacion.csv listo ·", len(out), "estaciones")

## Empaquetar y descargar

In [ ]:
import zipfile
with zipfile.ZipFile("datos_nuevos.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in ["hourofweek_2024_enefeb.csv", "station_month_hour.csv.gz", "acs_por_estacion.csv"]:
        if os.path.exists(f): z.write(f); print("  +", f, f"{os.path.getsize(f)/1e6:.2f} MB")
print("\ndatos_nuevos.zip", f"{os.path.getsize('datos_nuevos.zip')/1e6:.2f} MB")
try:
    from google.colab import files; files.download("datos_nuevos.zip")
except Exception: pass